EDA - Datasets are extracted from: "[here](https://www.kaggle.com/datasets/nathansmallcalder/lol-match-history-and-summoner-data-80k-matches)"

Posibles hipótesis:
- La gente se toma más en serio las partidas ranked que las normales (comparar RankFk '0' con el resto)
- El oro total está más fuertemente asociado con la victoria que el KDA | La diferencia de oro entre equipos es mayor en partidas ganadas que la diferencia de KDA
- La diferencia de objetivos entre equipos es mayor que la diferencia en kills en partidas ganadas.


In [1]:
import os
import numpy as np
import pandas as pd
os.chdir("..")

from utils import clean_utils

Primero cargamos los 6 datasets

In [ ]:
# ID matching dataframes
df_champions = pd.read_csv("datasets/ChampionTbl.csv", sep=",")  # Links ChampionId (ChampionFk) and ChampionName
df_items = pd.read_csv("datasets/ItemTbl.csv", sep=",")  # Links ItemID and ItemName
df_rank = pd.read_csv("datasets/RankTbl.csv", sep=",")  # Links RankId (RankFk) and RankName 
df_summoner_match = pd.read_csv("datasets/SummonerMatchTbl.csv", sep=",")  # Links SummonerFk, MatchFk, and ChampionFk by SummonerMatchId (SummonerMatchFk)

# Match info dataframes
df_match = pd.read_csv("datasets/MatchTbl.csv", sep=",")  # Contains general stats for any MatchFk (Patch, QueueType, RankFk, GameDuration)
df_match_stats = pd.read_csv("datasets/MatchStatsTbl.csv", sep=",")  # Personal match summoner stats for any SummonerMatchFk (MinionsKilled, DmgDealt, Win/Loss, ...)
df_team_match_stats = pd.read_csv("datasets/TeamMatchTbl.csv", sep=",")  # Team match stats for any TeamID (MatchFk, B1Champ, Win/Loss, ...)

El análisis va a ser realizado sobre las partidas del modo clásico. 
Seleccionamos únicamente las partidas cuya *"QueueType"* sea *"CLASSIC"*.

In [3]:
classic_mask = df_match["QueueType"] == "CLASSIC"
df_classic_match = df_match[classic_mask]

*League of Legends* es un juego que es actualizado con un parche cada 2 semanas. Cada nueva versión modifica ligeramente factores del juego como las estadísticas de los campeones, objetos, objetivos, ... Debido a esto, el juego de hace varios parches puede no ser representativo de la versión actual. 

Para tener información lo más actualizada posible vamos a eliminar las partidas realizadas en versiones de los parches más antiguos. Cada valor de la columna "Patch" tiene sigue el siguiente formato --> "15.24.734.7485". 

Es necesario transformar esta columna: 
- Simplificarlo para que coincida con las versiones que lanza *Riot Games* de cara al público. 
- Transformarlo en float para poder ordenarlos (15.24).

Para aplicar estas 2 transformaciones le pasamos la columna al método `parse_patches()`. Tras ello, ordenamos las versiones por nº de partidas.

In [5]:
df_classic_match["Patch"] = clean_utils.parse_patches(df_classic_match["Patch"])
df_classic_match["Patch"].value_counts().sort_values(ascending=False)

Patch
15.24    55089
16.01    41581
15.23    28969
15.22    25031
15.20    21090
15.21     9802
15.19     3024
15.18     1169
15.17      778
15.16      415
15.14      405
15.13      370
15.15      328
15.12      193
14.23      187
15.11      175
14.24      172
15.09      171
15.08      161
15.07      158
14.13      152
15.10      110
14.16      107
14.21      107
15.04      102
14.22       88
15.03       85
15.06       85
15.02       78
14.20       69
15.05       68
14.15       67
15.01       65
14.18       49
14.14       46
14.03       40
14.17       40
14.12       39
14.19       39
14.06       15
14.05       12
14.08       12
14.11       11
14.07        8
14.10        8
14.02        6
13.23        5
13.24        3
13.22        2
14.04        2
Name: count, dtype: int64

Hay pocas partidas de los parches más antiguos, mientras que los parches más recientes tienen un mayor volumen de muestras.

Aprovechando esto, vamos a eliminar las partidas de los parches que tienen menos de 1000 partidas (que también son los más viejos).

In [12]:
# Count how many times each patch appears
patch_counts = df_classic_match["Patch"].value_counts()
# Transform the value of the 'Patch' column into its Patch count and compare it to 1000
n_minimum_games_mask = df_classic_match["Patch"].map(patch_counts) >= 1000 
df_recent_classic_match = df_classic_match[n_minimum_games_mask] 

df_recent_classic_match["Patch"].value_counts().sort_values(ascending=False)


Patch
15.24    55089
16.01    41581
15.23    28969
15.22    25031
15.20    21090
15.21     9802
15.19     3024
15.18     1169
Name: count, dtype: int64

Una vez que tenemos las partidas recientes y del modo clásico, podemos usar `merge()` para conseguir las estadísticas de las otras tablas. En el *dataframe* actual tenemos el *"MatchId"*.

Dependiendo de las estadísticas que busquemos debemos:
- Estadísticas **individuales**: Utiliza el *"SummonerMatchId"*, por lo que debemos pivotar en el *df* de la tabla *"SummonerMatchTbl.csv"* que contiene tanto *"MatchFk"* como *"SummonerMatchFk"*.
- Estadísticas **grupales**: Utiliza *"MatchFk"*, así que podemos combinarlas directamente.

In [17]:
# Estadísticas individuales
individual_data = pd.merge(df_recent_classic_match, df_summoner_match, how="inner", left_on="MatchId", right_on="MatchFk")
individual_data = pd.merge(individual_data, df_match_stats, how="inner", left_on="SummonerMatchId", right_on="SummonerMatchFk")

# Estadísticas grupales
grupal_data = pd.merge(df_recent_classic_match, df_team_match_stats, how="inner", left_on="MatchId", right_on="MatchFk")


¡Ya podemos empezar a trabajar sobre los datos de las partidas más recientes del modo que queremos!

In [18]:
individual_data["Lane"].value_counts()

Lane
BOTTOM     105897
JUNGLE      88370
MIDDLE      85187
TOP         80323
UTILITY     53599
NONE        22536
SUPPORT        12
Name: count, dtype: int64